# Fine-tuning + éval compost — version LOCALE (GPU)

Équivalent local de `colab_finetune.ipynb`, sans Colab ni Drive : tout tourne sur ton GPU et s'enregistre dans `runs/`.

**Prérequis** : venv installé, et le `best.pt` **pré-entraîné** récupéré sur ta machine.

**Utilisation** : règle les 2 variables dans la cellule *Config*, puis *Run All* (Kernel → Restart & Run All). Les étapes s'enchaînent toutes seules.

In [ ]:
# --- Config : les 2 seules choses à régler ---
PRETRAIN = "/home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/runs/train_29-06_17h13/weights/best.pt"   # modèle PRE-ENTRAINE (a changer selon le run)
DEVICE   = "0"          # "0" = GPU, "cpu" si pas de GPU
EPOCHS   = 30
BATCH    = 8            # baisse à 4 si erreur mémoire GPU

import os, glob, subprocess
os.chdir(os.path.expanduser("~/stage/Compost_Waste_Yolo/compost-yolo"))
PYTHON = os.path.join(os.getcwd(), "venv", "bin", "python")   # python du venv (contient compost_detection)
# le kernel Jupyter exporte MPLBACKEND=...inline, invalide hors notebook -> backend fichier
ENV = {**os.environ, "MPLBACKEND": "Agg"}
def run(cmd):
    if cmd.startswith("python "):
        cmd = PYTHON + cmd[6:]          # force le python du venv, quel que soit le kernel Jupyter
    print("\n$", cmd)
    if subprocess.call(cmd, shell=True, env=ENV) != 0:
        raise RuntimeError("échec de : " + cmd)
print("dossier :", os.getcwd(), "| python :", PYTHON)

## 1. Split — met de côté le test compost + prépare le train/val du fine-tuning

In [ ]:
run("python scripts/split_captures.py --source data/raw/captures --output data/finetune")
run("python scripts/prepare_dataset.py --source data/finetune/captures_finetune "
    "--output data/finetune/dataset_finetune --ratios 0.85 0.15 0")

## 2. Éval B — le modèle pré-entraîné sur le test compost (AVANT fine-tuning)

In [ ]:
run(f"python scripts/evaluate.py --weights {PRETRAIN} "
    f"--data data/finetune/captures_test/data.yaml --split test --device {DEVICE}")

## 3. Fine-tuning — repart du pré-entraîné, learning rate bas

In [ ]:
run(f"python scripts/train.py --model {PRETRAIN} "
    f"--data data/finetune/dataset_finetune/data.yaml "
    f"--epochs {EPOCHS} --lr0 0.001 --batch {BATCH} --device {DEVICE}")

## 4. Éval C — le modèle fine-tuné sur le MÊME test compost (APRÈS)
Le dernier `runs/train_*` est trouvé automatiquement.

In [ ]:
finetuned = max(glob.glob("runs/train_*/weights/best.pt"), key=os.path.getmtime)
print("modèle fine-tuné :", finetuned)
run(f"python scripts/evaluate.py --weights {finetuned} "
    f"--data data/finetune/captures_test/data.yaml --split test --device {DEVICE}")
print("\nTerminé. Compare les deux derniers dossiers runs/eval_* : B (avant) vs C (après).")